<a href="https://colab.research.google.com/github/chewanna7-code/NatureInsightStudy/blob/main/Bucket_Isolation_Extractor_Cleaned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Bucket Isolation Extraction and Hydrological Summary

## Purpose

This notebook extracts peak flow and time-to-peak results from NatureInsight® hydrograph export files.

It was used to compare the hydrological contribution of different NatureInsight® storage buckets and individual intervention scenarios across return periods.

---

## What This Notebook Does

- Uploads multiple NatureInsight® Excel hydrograph exports.
- Reads scenario information from each file name.
- Extracts peak flow and time to peak from each hydrograph.
- Compares each scenario against the Wansbeck baseline flow values.
- Calculates percentage peak flow reduction.
- Calculates percentage change in time to peak.
- Creates summary and pivot tables.
- Exports the processed results to Excel.

---

## File Naming Assumption

The code expects filenames to contain scenario information in a consistent format, for example:

```text
R1-SS70-10Y-RUN-WAN.xlsx
R1-SS70-50Y-FPR-WAN.xlsx
R1-SS70-100Y-RAF-WAN.xlsx
```

The filename is used to extract:

- Rank
- Suitability Score
- Return Period
- Bucket or intervention type
- Catchment

---

## Dissertation Context

This workflow supports the hydrological results section by helping isolate how different NatureInsight® storage categories affect modelled flood response.

It was particularly useful for identifying whether runoff storage, floodplain storage and land-use storage contributed differently to peak flow reduction and time-to-peak delay.



## Step 1 – Import Required Libraries

This section loads the packages required to:

- Read Excel files.
- Process uploaded files in Google Colab.
- Extract values from NatureInsight® hydrograph spreadsheets.
- Build summary tables.


In [ ]:

# Import libraries needed for file upload, Excel reading and data processing

import pandas as pd
import io
from google.colab import files
from openpyxl import load_workbook



## Step 2 – Parse Scenario Information from File Names

The NatureInsight® exports are named using scenario identifiers.

This function reads each filename and extracts the key information needed for the summary table, including:

- Rank
- Suitability Score
- Return Period
- Bucket or intervention type
- Catchment

If no bucket name is found, the file is treated as a baseline scenario.


In [ ]:

# Function to extract scenario metadata from each uploaded filename

def parse_filename(filename):
    """Extract scenario information from a NatureInsight® export filename."""

    # Standardise filename text before splitting
    name = filename.replace(".xlsx", "").replace(".XLSX", "").upper()
    parts = name.split("-")

    # Set up blank metadata fields
    result = {
        "Filename": filename,
        "Rank": None,
        "SS": None,
        "RP (years)": None,
        "Bucket": None,
        "Catchment": None,
    }

    # Identify scenario components from filename sections
    for part in parts:
        if part.startswith("SS") and part[2:].isdigit():
            result["SS"] = int(part[2:])

        elif part.startswith("R") and part[1:].isdigit():
            result["Rank"] = int(part[1:])

        elif part.endswith("Y") and part[:-1].isdigit():
            result["RP (years)"] = int(part[:-1])

        elif part in ["WAN", "CAL", "EDEN"]:
            result["Catchment"] = part

        elif part in ["RUN", "FPR", "SM", "RAF", "FR", "LU", "ALL", "BASE", "BASELINE"]:
            result["Bucket"] = part

    # If no bucket/intervention type is detected, classify as baseline
    if result["Bucket"] is None:
        result["Bucket"] = "Baseline"

    return result



## Step 3 – Extract Peak Flow and Time to Peak

This function opens each NatureInsight® Excel export and searches for the columns containing:

- Total time
- Total flow

It then identifies the maximum flow value and records the time at which that peak occurs.


In [ ]:

# Function to extract hydrological metrics from a NatureInsight® Excel export

def extract_metrics(file_bytes, filename):
    """Extract peak flow and time-to-peak from a NatureInsight® hydrograph export."""

    # Open uploaded Excel file directly from memory
    wb = load_workbook(io.BytesIO(file_bytes), data_only=True)
    ws = wb.active

    total_time_col = None
    total_flow_col = None
    header_row = None

    # Search worksheet for the relevant hydrograph columns
    for row in ws.iter_rows():
        for cell in row:
            val = str(cell.value).strip().lower() if cell.value else ""

            if "total" in val and "time" in val and "sec" in val:
                total_time_col = cell.column
                header_row = cell.row

            if "total" in val and "flow" in val and "time" not in val:
                total_flow_col = cell.column

        if total_time_col and total_flow_col:
            break

    # Return empty values if the expected columns cannot be found
    if not total_time_col or not total_flow_col:
        print(f"WARNING: Could not find required columns in {filename}")
        return {
            "Peak Flow (m³/s)": None,
            "Time to Peak (sec)": None,
            "Time to Peak (hrs)": None,
        }

    # Read time and flow values below the header row
    times = []
    flows = []

    for row_index in range(header_row + 1, ws.max_row + 1):
        time_value = ws.cell(row_index, total_time_col).value
        flow_value = ws.cell(row_index, total_flow_col).value

        if time_value is not None and flow_value is not None:
            try:
                times.append(float(time_value))
                flows.append(float(flow_value))
            except (TypeError, ValueError):
                pass

    # Return empty values if no flow data is found
    if not flows:
        print(f"WARNING: No flow data found in {filename}")
        return {
            "Peak Flow (m³/s)": None,
            "Time to Peak (sec)": None,
            "Time to Peak (hrs)": None,
        }

    # Identify the peak flow and the corresponding time
    peak_flow = max(flows)
    peak_index = flows.index(peak_flow)
    time_to_peak_sec = times[peak_index]
    time_to_peak_hrs = round(time_to_peak_sec / 3600, 4)

    return {
        "Peak Flow (m³/s)": round(peak_flow, 4),
        "Time to Peak (sec)": time_to_peak_sec,
        "Time to Peak (hrs)": time_to_peak_hrs,
    }



## Step 4 – Upload and Process Hydrograph Files

Upload all relevant NatureInsight® hydrograph Excel files at once.

The notebook will loop through each file, extract the scenario metadata and hydrological metrics, then store everything in a single results table.


In [ ]:

# Upload all relevant NatureInsight® hydrograph Excel exports

print("Upload all bucket isolation files now...")
uploaded = files.upload()

# Process each uploaded file
rows = []

for filename, file_bytes in uploaded.items():
    print(f"\nProcessing: {filename}")

    # Extract scenario details from filename
    metadata = parse_filename(filename)

    # Extract peak flow and time-to-peak from the file
    metrics = extract_metrics(file_bytes, filename)

    # Combine metadata and extracted metrics
    rows.append({**metadata, **metrics})

    # Print a quick progress summary
    print(
        f"  Bucket={metadata['Bucket']}, "
        f"RP={metadata['RP (years)']}yr | "
        f"Peak={metrics['Peak Flow (m³/s)']} m³/s | "
        f"TTP={metrics['Time to Peak (hrs)']} hrs"
    )



## Step 5 – Build Summary Table and Calculate Reductions

This section compares each extracted scenario against the Wansbeck baseline values.

The baseline values are entered manually from the existing NatureInsight® baseline results.

The calculated metrics are:

- Percentage peak flow reduction.
- Percentage change in time to peak.

Positive peak reduction values indicate reduced peak flow compared with the baseline.


In [ ]:

# Build dataframe from extracted results

df = pd.DataFrame(rows).sort_values(
    ["Catchment", "RP (years)", "Bucket"]
).reset_index(drop=True)

# Wansbeck baseline peak flows from NatureInsight® baseline outputs
wan_baseline_peak_flow = {
    10: 109.6069,
    20: 125.1541,
    50: 147.2691,
    100: 165.4509,
    200: 185.0643,
    500: 213.4386,
}

# Wansbeck baseline time-to-peak values in seconds
wan_baseline_ttp = {
    10: 83400,
    20: 83400,
    50: 83400,
    100: 83400,
    200: 83400,
    500: 83400,
}

# Match baseline values to each scenario return period
df["Baseline Peak Flow"] = df["RP (years)"].map(wan_baseline_peak_flow)
df["Baseline TTP (sec)"] = df["RP (years)"].map(wan_baseline_ttp)

# Calculate percentage peak flow reduction relative to baseline
df["% Peak Reduction"] = df.apply(
    lambda row: round(
        (row["Baseline Peak Flow"] - row["Peak Flow (m³/s)"])
        / row["Baseline Peak Flow"] * 100,
        2,
    )
    if pd.notna(row["Baseline Peak Flow"]) and pd.notna(row["Peak Flow (m³/s)"])
    else None,
    axis=1,
)

# Calculate percentage change in time-to-peak relative to baseline
df["% Slower (TTP)"] = df.apply(
    lambda row: round(
        (row["Time to Peak (sec)"] - row["Baseline TTP (sec)"])
        / row["Baseline TTP (sec)"] * 100,
        2,
    )
    if pd.notna(row["Baseline TTP (sec)"]) and pd.notna(row["Time to Peak (sec)"])
    else None,
    axis=1,
)

# Remove helper baseline columns from final display
df = df.drop(columns=["Baseline Peak Flow", "Baseline TTP (sec)"])

print("\nFULL SUMMARY TABLE:")
display(df)



## Step 6 – Create Pivot Tables

The pivot tables make the results easier to compare across return periods.

Two summary tables are produced:

1. Percentage peak flow reduction by bucket and return period.
2. Peak flow by bucket and return period.


In [ ]:

# Pivot table showing percentage peak flow reduction

print("\n% PEAK REDUCTION BY BUCKET AND RETURN PERIOD:")

pivot_reduction = df.pivot_table(
    index="Bucket",
    columns="RP (years)",
    values="% Peak Reduction",
)

display(pivot_reduction)


# Pivot table showing peak flow values

print("\nPEAK FLOW BY BUCKET AND RETURN PERIOD:")

pivot_peak_flow = df.pivot_table(
    index="Bucket",
    columns="RP (years)",
    values="Peak Flow (m³/s)",
)

display(pivot_peak_flow)



## Step 7 – Export Results

The final results are exported to an Excel workbook containing:

- Full summary table.
- Percentage peak flow reduction pivot.
- Peak flow pivot.

This output can then be used in dissertation tables, graphs or additional analysis.


In [ ]:

# Export all summary outputs to Excel

output_file = "Bucket_Isolation_Summary.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Full Summary", index=False)
    pivot_reduction.to_excel(writer, sheet_name="% Peak Reduction Pivot")
    pivot_peak_flow.to_excel(writer, sheet_name="Peak Flow Pivot")

files.download(output_file)

print(f"\nSaved: {output_file}")
print("Done!")
